# Chat data: the message formatter

A chat model is sampled in its own message format: the special tokens and layout that turn a list of messages into one token sequence. Fine-tuning data has to be written in the same format, or the model is trained on prompts it never meets when it is sampled. The cookbook has a message formatter for each model family, which does both sides. In this tutorial you build one, write a prompt with it, turn conversations into datums, look at what each one trains, meet Qwen's thinking switch, read a reply back as a message, and format a tool call.

It needs no service and no API key: everything here runs on your machine. It uses the cookbook's package, `looper_cookbook`. Until that is published, run this notebook from a checkout of the repository, in its environment.

## Settings

The model, whose tokenizer and formatter the notebook uses, and the longest datum a run will take.

In [1]:
MODEL = "Qwen/Qwen3.5-9B"
MAX_SEQ_LEN = 2048

## A formatter and a prompt

`message_formatter` builds the formatter of a model Looper serves, over the model's tokenizer. Each family has its own: Qwen3.5's here, and others for Qwen3.8 and Muse Glimmer. `enable_thinking=False` is a setting of Qwen's format: the model answers without reasoning first.

`build_generation_prompt` writes a conversation, then the opening of the next assistant turn: the token ids the model is given to write that turn, which is what a sampler takes. `stop_token_ids` is the token that ends the turn, for the sampler to stop on.

In [2]:
from transformers import AutoTokenizer

from looper_cookbook.message_formatters.models import message_formatter

tokenizer = AutoTokenizer.from_pretrained(MODEL)
formatter = message_formatter(MODEL, tokenizer, enable_thinking=False)

question = [{"role": "user", "content": "Tell me about the red fox."}]
prompt = formatter.build_generation_prompt(question)
print(type(formatter).__name__)
print(len(prompt), "tokens:")
print(tokenizer.decode(prompt))
print("a turn ends on", tokenizer.decode(formatter.stop_token_ids))

Qwen3_5MessageFormatter
19 tokens:
<|im_start|>user
Tell me about the red fox.<|im_end|>
<|im_start|>assistant
<think>

</think>


a turn ends on <|im_end|>


## One conversation, one datum per assistant turn

A conversation is a list of chat messages. `build_supervised_examples` writes it in the same format and returns one datum for each assistant turn.

In [3]:
conversation = [
    *question,
    {
        "role": "assistant",
        "content": "Habitat: forests, grasslands and cities.\nDiet: rodents, birds and fruit.",
    },
]
[datum] = formatter.build_supervised_examples(conversation, max_seq_len=MAX_SEQ_LEN)
print(len(datum.tokens), "tokens,", int(sum(datum.weights)), "of them targets")

41 tokens, 22 of them targets


## What a datum trains

A datum's tokens are its turn's prompt, and then the turn. Its weights split them in two:

- **Context**, weight 0: everything up to where the assistant starts to write. This is exactly the prompt above, the one the model is given when it is sampled for this turn.
- **Targets**, weight 1: the assistant's turn, through the end token. Stopping is part of what the model learns.

In [4]:
context = [token for token, weight in zip(datum.tokens, datum.weights) if not weight]
print("the context is the prompt:", context == prompt)
print()
print("targets:")
for text in formatter.targets(datum):
    print(repr(text))

the context is the prompt: True

targets:
'Habitat: forests, grasslands and cities.\nDiet: rodents, birds and fruit.<|im_end|>'


## Several turns

A conversation with two assistant turns makes two datums. The second holds the first exchange as context and trains only the second answer.

Why not one datum for the whole conversation? A format may write a turn differently once later turns follow it. Qwen3.5's does: it keeps a turn's reasoning block only after the last user message. Trained as one sequence, an early turn would follow a prompt the model never sees when it is sampled. A datum per turn trains every turn after exactly its sampled prompt.

In [5]:
longer = [
    *conversation,
    {"role": "user", "content": "And the Arctic fox?"},
    {
        "role": "assistant",
        "content": "Habitat: the tundra and sea ice.\nDiet: lemmings, birds and carrion.",
    },
]
datums = formatter.build_supervised_examples(longer, max_seq_len=MAX_SEQ_LEN)
for number, datum in enumerate(datums, start=1):
    print(f"datum {number}: {len(datum.tokens)} tokens, trains {formatter.targets(datum)}")

datum 1: 41 tokens, trains ['Habitat: forests, grasslands and cities.\nDiet: rodents, birds and fruit.<|im_end|>']
datum 2: 80 tokens, trains ['Habitat: the tundra and sea ice.\nDiet: lemmings, birds and carrion.<|im_end|>']


## The thinking switch

A Qwen formatter built without `enable_thinking=False` is in thinking mode: its prompt opens a reasoning block, and the model reasons before it answers. A conversation whose assistant turns carry `reasoning_content` trains in this mode, and the reasoning is part of the targets. The block's opening `<think>` is in the prompt, so the targets start inside it.

In [6]:
thinking = message_formatter(MODEL, tokenizer)
reasoned = [
    {"role": "user", "content": "Is a red fox bigger than an Arctic fox?"},
    {
        "role": "assistant",
        "reasoning_content": "Red foxes weigh up to 14 kg. Arctic foxes weigh up to 9 kg.",
        "content": "Yes. The red fox is the larger of the two.",
    },
]
print(repr(tokenizer.decode(thinking.build_generation_prompt(reasoned[:1]))))
[datum] = thinking.build_supervised_examples(reasoned, max_seq_len=MAX_SEQ_LEN)
for text in thinking.targets(datum):
    print(repr(text))

'<|im_start|>user\nIs a red fox bigger than an Arctic fox?<|im_end|>\n<|im_start|>assistant\n<think>\n'
'Red foxes weigh up to 14 kg. Arctic foxes weigh up to 9 kg.\n</think>\n\nYes. The red fox is the larger of the two.<|im_end|>'


The two modes prompt the model differently, so train and sample a model in the same one. With thinking off the prompt closes an empty reasoning block itself, which tells the model not to reason. A turn that carries reasoning cannot follow that prompt, and the formatter refuses it instead of training something approximate. The error names the turn and the fix.

A conversation without reasoning trains in either mode. In thinking mode its turn closes the block the prompt opened, empty.

In [7]:
try:
    formatter.build_supervised_examples(reasoned, max_seq_len=MAX_SEQ_LEN)
except ValueError as error:
    print(error)
print()
[plain] = thinking.build_supervised_examples(conversation, max_seq_len=MAX_SEQ_LEN)
print(thinking.targets(plain))

messages[1] of the conversation starting 'Is a red fox bigger than an Arctic fox?': this assistant turn carries reasoning, and the formatter has thinking off: its prompt closes an empty <think> block, which tells the model not to reason. Build the formatter with enable_thinking=True, or drop the turn's reasoning

['\n</think>\n\nHabitat: forests, grasslands and cities.\nDiet: rodents, birds and fruit.<|im_end|>']


## A reply read back

A sampler returns a turn as token ids. `parse_response` reads them back as an assistant message, with the turn's reasoning as `reasoning_content`, apart from its `content`. It also says whether the turn ended on its end token, which a reply cut off at `max_tokens` did not. A reply cut off while it reasoned has no content yet.

Here the datum's targets stand in for a sampled reply.

In [8]:
reply = datum.tokens[list(datum.weights).index(1) :]
message, ended = thinking.parse_response(reply)
print(message)
print("ended on its end token:", ended)
print("cut short:", thinking.parse_response(reply[:8]))

{'role': 'assistant', 'reasoning_content': 'Red foxes weigh up to 14 kg. Arctic foxes weigh up to 9 kg.', 'content': 'Yes. The red fox is the larger of the two.'}
ended on its end token: True
cut short: ({'role': 'assistant', 'reasoning_content': 'Red foxes weigh up to 1', 'content': ''}, False)


## Tool calls

A model that calls tools is given their JSON schemas, in the `tools` format chat APIs use. Pass them with the conversation, and the formatter lists them where the model's format does. A turn that calls a tool carries `tool_calls`, and a tool's reply is a message with the role `tool`.

In a conversation you train, a turn that calls a tool is a datum whose targets are its call. In a rollout, `parse_response` reads a sampled turn's calls back as `tool_calls`, with each call's arguments as a mapping. Give it the tools too, and each argument has the type its tool declares; without them every argument is the text the model wrote.

In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "weigh",
            "description": "An animal's largest weight, in kilograms.",
            "parameters": {
                "type": "object",
                "properties": {"animal": {"type": "string"}},
                "required": ["animal"],
            },
        },
    }
]
call = {"type": "function", "function": {"name": "weigh", "arguments": {"animal": "red fox"}}}
looked_up = [
    {"role": "user", "content": "How heavy is a red fox?"},
    {"role": "assistant", "content": "", "tool_calls": [call]},
    {"role": "tool", "name": "weigh", "content": "14"},
    {"role": "assistant", "content": "Up to 14 kg."},
]
called, answered = formatter.build_supervised_examples(looked_up, tools, max_seq_len=MAX_SEQ_LEN)
print(formatter.targets(called))
print(formatter.targets(answered))
print()
sampled = called.tokens[list(called.weights).index(1) :]
print(formatter.parse_response(sampled, tools))

['<tool_call>\n<function=weigh>\n<parameter=animal>\nred fox\n</parameter>\n</function>\n</tool_call><|im_end|>']
['Up to 14 kg.<|im_end|>']

({'role': 'assistant', 'content': '', 'tool_calls': [{'type': 'function', 'function': {'name': 'weigh', 'arguments': {'animal': 'red fox'}}}]}, True)


## A file of conversations

`read_conversations` reads a JSONL file with one conversation per line, as `{"messages": [...]}`. A line may also carry `tools`, which the formatter takes as its second argument.

The cookbook ships a small example file: 16 conversations that each answer a question about an animal with a three-line card. Two of them have a follow-up question, so they make two datums each.

In [10]:
from importlib import resources

from looper_cookbook.datasets.conversations import read_conversations

path = resources.files("looper_cookbook") / "example_data" / "conversations.jsonl"
conversations = read_conversations(path)
data = [
    datum
    for conversation in conversations
    for datum in formatter.build_supervised_examples(
        conversation["messages"], conversation.get("tools"), max_seq_len=MAX_SEQ_LEN
    )
]
tokens = sum(len(datum.tokens) for datum in data)
trained = sum(int(sum(datum.weights)) for datum in data)
print(f"{len(conversations)} conversations: {len(data)} datums, {tokens} tokens, {trained} targets")

16 conversations: 18 datums, 1366 tokens, 805 targets


## What else is refused

A formatter fails a conversation rather than build a datum its model's format cannot hold:

- a turn that starts past `max_seq_len`. A turn that only ends past it is cut there: a datum keeps its head, so what the model conditions on never changes;
- a conversation with no assistant turn, since nothing in it trains;
- content that is not text;
- a tokenizer of another model, when the formatter is built.

Each family's formatter is a module of `looper_cookbook/message_formatters/`, written from the model's chat template and short enough to read.

## Next steps

- [The SFT loop](301_sft_loop.ipynb) trains these datums and samples the result.